# 3c. Anonymization checks

On the original text (step 2), after part 1 (rules) and after part 2 (final text):
- distinctive outlet names left as a word (as in step 3);
- an **LLM attacker** (Mistral-Small-24B-Instruct, 4 bit; larger than the detector) asked which website published the
  article, on 3 articles per outlet (678): share of right domains;
- a **classifier** that tries to recognize the outlet among the 226 (TF-IDF + linear SVM, 80/20 of every outlet's
  articles). The writing style stays (it is what the paper studies), so this does not go to zero;
- the **agreement** of the detector with the larger model on 400 articles: outlet phrases the larger one finds that are
  still in the final text;
- a **sheet for a check by hand** of 100 random articles (`anonymization/audit.html`, to fill `audit.csv`).

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import json

import anonymization as AN
import anonymization_llm as AL

raw = pd.read_csv(C.OUT / "articles.csv.gz")
part1 = pd.read_csv(C.OUT / "articles_rules.csv.gz")
art = pd.read_csv(C.OUT / "articles_anonymous.csv.gz")
assert (raw.article_id.to_numpy() == art.article_id.to_numpy()).all() and (part1.article_id.to_numpy() == art.article_id.to_numpy()).all()
versions = {"original": raw, "rules (part 1)": part1, "final (part 2)": art}

In [ ]:
_, _, found = AN.anonymize(raw)
for v, d in versions.items():
    left = AN.leftovers(d, found)
    print(f"{v}: distinctive names left as a word: {left[~left.common_word].left_as_word.sum()}")

In [ ]:
S = AL.sample(art)
f = AL.WORK / "attack_answers.json"
att = json.load(open(f)) if f.exists() else {}
if any(v not in att for v in versions):
    import models as MO
    MO.wait_for_gpu(18, AL.LOG)
    try:                               # the attacker is the larger model (stronger); the detector if it is missing
        tok, m = AL.load_llm(AL.CHECKER, four_bit=True); att["attacker"] = AL.CHECKER; budget, mb = 12000, 8
    except Exception as e:
        print("larger model missing, the attacker is the detector:", e)
        tok, m = AL.load_llm(AL.DETECTOR); att["attacker"] = AL.DETECTOR; budget, mb = 24000, 32
    for v, d in versions.items():
        if v not in att:
            prompts = [AL.ATTACK.format(title=str(d.title.fillna("")[i]), text=" ".join(str(d.text.fillna("")[i]).split()[:1500])) for i in S]
            att[v] = [a for _, a in sorted(AL.generate(tok, m, prompts, 12, budget=budget, max_batch=mb).items())]
            json.dump(att, open(f, "w"))
    del m; import torch; torch.cuda.empty_cache()
print("attacker:", att.get("attacker"))
checks = {"LLM attacker names the website": {v: AL.attack_accuracy(att[v], art.domain[S]) for v in versions},
          "classifier recognizes the outlet (226)": {v: AL.outlet_classifier(d.text.fillna("").to_numpy(), art.domain.to_numpy())
                                                     for v, d in versions.items()}}
T = pd.DataFrame(checks).T; T.to_csv(AL.WORK / "checks.csv"); T.round(3)

In [ ]:
f = AL.WORK / "checker_answers.json"
idx = art.sample(n=AL.CHECK_SAMPLE, random_state=C.SEED).index
try:                                   # the larger model is only a check: if it is missing, the step goes on
    if not f.exists():
        import models as MO
        MO.wait_for_gpu(18, AL.LOG)
        tok, m = AL.load_llm(AL.CHECKER, four_bit=True)
        ch = AL.generate(tok, m, AL.detection_prompts(part1.loc[idx].reset_index(drop=True)), 160, budget=12000, max_batch=8)
        json.dump({str(int(idx[k])): v for k, v in ch.items()}, open(f, "w"))
        del m; import torch; torch.cuda.empty_cache()
    checker = {int(k): v for k, v in json.load(open(f)).items()}
    Lc = AL.lexicon(part1, checker)
    phrases = set(map(tuple, Lc[Lc.kept & (Lc.kind == "outlet")][["domain", "phrase"]].to_numpy()))
    left = sorted(p for d, p in phrases if any(AL._word_re(p, True).search(t) for t in art.text[art.domain == d].fillna("")))
    print(f"on {len(idx)} articles the larger model finds {len(phrases)} outlet phrases; still in the final text: {len(left)}")
    pd.DataFrame({"phrase": left}).to_csv(AL.WORK / "checker_left.csv", index=False); print(left[:40])
except Exception as e:
    print("check with the larger model not done:", type(e).__name__, e)

In [ ]:
A = art.sample(n=AL.AUDIT, random_state=C.SEED + 1)
AL.audit_sheet(part1.text.fillna("")[A.index], A.text.fillna(""), A.article_id, A.domain, AL.WORK / "audit.html")
pd.DataFrame({"article_id": A.article_id, "identifiers left": "", "wrong replacements": ""}).to_csv(AL.WORK / "audit.csv", index=False)
print("for the check by hand:", AL.WORK / "audit.html", "and", AL.WORK / "audit.csv")